In [1]:
#参考https://github.com/bbruceyuan/LLMs-Zero-to-Hero/blob/master/src/video/build_gpt.ipynb
# import packages
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass

import math

torch.manual_seed(1)

In [2]:
@dataclass

class DataConfig:
    block_size: int = 64

In [3]:
class SingleHeadAttention(nn.Module):
    # 单头注意力
    def __init__(self, config):
        super().__init__()
        self.key = nn.Linear(config.n_embd, config.head_size)
        self.value = nn.Linear(config.n_embd, config.head_size)
        self.query = nn.Linear(config.n_embd, config.head_size)
        self.head_size = config.head_size

        
        # need to understand!!!
        self.register_buffer(
            'attention_mask',
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            ))
        # 设置掩码的一串代码
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()
        # 运用Module模块自带的线性层设置参数与进行矩阵乘法
        k = self.key(x)
        v = self.value(x)
        q = self.query(x)

        weight = q @ k.transpose(-2, -1)

        weight = weight.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0,
            float('-inf')
        ) / math.sqrt(self.head_size) # 这里的 hidden_size 其实是 head_size，因为是单头
        weight = F.softmax(weight, dim=-1)
        weight = self.dropout(weight)
        out = weight @ v
        return out 

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        output = self.proj(output)
        output = self.dropout(output)
        return output      

In [5]:
class FeedForward(nn.Module):
    # 实际上为MLP
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.att = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [7]:
'''toy'''
weight = torch.rand(3, 4)
a = torch.tensor([1, 1])
print(a)
print(weight)
print(weight[a])

tensor([1, 1])
tensor([[0.7576, 0.2793, 0.4031, 0.7347],
        [0.0293, 0.7999, 0.3971, 0.7544],
        [0.5695, 0.4388, 0.6387, 0.5247]])
tensor([[0.0293, 0.7999, 0.3971, 0.7544],
        [0.0293, 0.7999, 0.3971, 0.7544]])


In [8]:
'''Parameters nn.embedding()中的参数
1.num_embeddings (int) – size of the dictionary of embeddings
2.embedding_dim (int) – the size of each embedding vector
3.padding_idx (int, optional) – If specified, the entries at padding_idx do not contribute to the gradient; therefore,
the embedding vector at padding_idx is not updated during training, i.e. it remains as a fixed “pad”.
For a newly constructed Embedding, the embedding vector at padding_idx will default to all zeros, 
but can be updated to another value to be used as the padding vector.这个参数本项目不需要
4.max_norm (float, optional) – If given, each embedding vector with norm larger than max_norm is renormalized to have norm max_norm.
5.norm_type (float, optional) – The p of the p-norm to compute for the max_norm option. Default 2.
6.scale_grad_by_freq (bool, optional) – If given, this will scale gradients by the inverse of frequency of the words in the mini-batch. Default False.
7.sparse (bool, optional) – If True, gradient w.r.t. weight matrix will be a sparse tensor. See Notes for more details regarding sparse gradients.
from https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html
'''
#后面的参数是训练时需要的以避免一些极端情况， 可以先不管后面的参数

class TokenEmbedding(nn.Module):
    # 为了继承nn.Module中foward函数的用法,更加方便，代码更加简洁
    def __init__(self, vocab_size: int, n_embd: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd

        #初始化weight，用nn自带的训练
        self.weight = nn.Parameter(torch.Tensor(vocab_size, n_embd))
        #nn.init.xavier_uniform_(self.weight) 在后面初始化了
        #参考https://zhuanlan.zhihu.com/p/630487545，选择矩阵的初始化方法
    
    def forward(self, idx):
        #idx为一个二维矩阵，每一行为一个句子，每一行中的各个数字代表token的id，也是12×block_size,已经被填充好，但这样写更加通用
        #tensor的作用
        #将idx中的token数目转化为了一个向量
        return self.weight[idx]    


class PositionalEmbedding(nn.Module):
    def __init__(self, block_size: int, n_embd:int):
        super().__init__()
        self.block_size = block_size
        self.n_embd = n_embd #有点累赘，但懒得改了
        self.pe = torch.zeros(block_size, n_embd) 

        self.pos = torch.arange(0, block_size).unsqueeze(1)
        div_term = torch.exp( torch.arange(0, n_embd, 2) * - (math.log(10000)) / n_embd)
        self.pe[:, 0::2] = torch.sin(self.pos * div_term)
        self.pe[:, 1::2] = torch.cos(self.pos * div_term)
        #构建 block_size×n_embd的位置矩阵, positionalembedding

    def forward(self, pos):
        #pos为一个二维矩阵，12×block_size
        return self.pe[pos]


In [9]:
# 写一个 dataset，为了 Dataloader 准备
class MyDataset(Dataset):
    def __init__(self, block_size=512):
        self.block_size = block_size


        self.encoded_data = []

        self.wordstonum = {}
        self.wordstonum[''] = 0
        self.numtowords = {}
        self.numtowords[0]  = ''
        self.total_words = 0

        self.max_lines = 100000
        raw_data = []

        import os
        self.path = r"C:\Users\余全霖\Desktop\人工智能导论\pj\dataset"
        files = os.listdir(self.path)
        
        for file in files:
            file_path = os.path.join(self.path, file)
            with open(file_path, 'r') as f:
                for i, line in enumerate(f):
                    if i >= self.max_lines:
                        break
                    text = line.strip()
                    if text != '':
                        raw_data.append(text)
                        for token in text:
                            if token not in self.wordstonum.keys():
                                self.total_words += 1
                                self.wordstonum[token] = self.total_words
                                self.numtowords[self.total_words] = token
        self.total_words += 1                       

        full_encoded = []
        for text in raw_data:
            encoded_text = self.encode(text)
            full_encoded.extend(encoded_text + [0])

            
        # 将长文本分割成训练样本
        for i in range(0, len(full_encoded), self.block_size):
            #多取一个token作为目标
            chunk = full_encoded[i:i+self.block_size+1]
            # 如果长度不够，用 eos_token 填充，所以所有输入的长度均为block_size
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [0] * (self.block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)

        #这一步已经做到了将不同长度的句子给padding了
    
    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        """将文本编码为token IDs"""
        ids = []
        for token in text:
            ids.append(self.wordstonum[token])
        
        return ids

    def decode(self, ids):
        """将token IDs解码为文本"""
        text = []
        for num in ids:
            text.append(self.numtowords[num])
        
        return text

In [10]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        # 这里暂时先用nn中的embedding模块， 后需要自主实现
        self.token_embedding_table = TokenEmbedding(config.vocab_size, config.n_embd)
        self.position_embeding_table = PositionalEmbedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)
        self.block_size = config.block_size
        self.batch_size = config.batch_size

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, TokenEmbedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx 为输入的 token ids
        # print(idx)
        batch, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embeding_table(
            torch.arange(seq_len, device=idx.device)
        )

        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, words_dict, text, max_length):

        idx = []
        seq = []
        for token in text:
            seq.append(words_dict.wordstonum[token])
        if len(text) < self.block_size:
            seq = [0] * (self.block_size - len(text)) + seq
        for _ in range(self.batch_size):
            idx.append(seq)

        idx = torch.tensor(idx)

        for _ in range(max_length):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]  # becomes (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)

        idx = idx[:, self.block_size - len(text):]

        for i in range(self.batch_size):
            for token in idx[i]:
                print(words_dict.numtowords[int(token)], end = '')
            print("\n")
            
        return 0

        

In [11]:
train_dataset = MyDataset(DataConfig.block_size)
words_dict = train_dataset
print(train_dataset.encode("萧"))
print(train_dataset.decode([164, 238, 0]))
print(train_dataset.total_words)

train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)

print(val_loader)

[3624]
['”', '避', '']
4470


In [12]:
@dataclass
class GPTConfig:
    block_size: int = DataConfig.block_size
    batch_size: int = 12
    n_layer: int = 6
    n_head: int = 12
    n_embd: int = 768
    head_size: int = n_embd // n_head
    dropout: float = 0.1

    vocab_size: int = words_dict.total_words

In [13]:
model = GPT(GPTConfig())
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 打印模型参数

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6} M")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

Total parameters: 49.394688 M


In [ ]:
# 训练循环
def train(model, optimizer, scheduler, train_loader, val_loader, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将数据移到设备上
        x, y = x.to(device), y.to(device)
        
        # 前向传播
        logits, loss = model(x, targets=y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 调整学习率
        scheduler.step()
        
        total_loss += loss.item()

        model.generate(words_dict, "萧炎", 10)
        
        print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
    return total_loss

def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss


for epoch in range(4):
    train_loss = train(model, optimizer, scheduler, train_loader, val_loader, device)
    val_loss = eval(model, val_loader, device)
    print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }
    # 保存每个epoch的模型
    torch.save(checkpoint, f'C:\\Users\\余全霖\\Desktop\\人工智能导论\\pj\\models\\model_epoch_{epoch}.pt')

萧炎纲，撕涵桃蝌，他<曦

萧炎，你晋搂俨涝嚷惧酋刑

萧炎肿蹬岵，揣吹，夸个睥

萧炎箭嗅9i陷掩褒翔赚躇

萧炎逝豹吹魃罐缈他俗渭，

萧炎余，，』瞧啸赳晌狱c

萧炎捷拿噬扩缉偌獠旋决骑

萧炎衣牌甘泳鹬煅爷骇去恒

萧炎，吞鹤熟物不瑰味咂行

萧炎毁谛，嫉满殁愁哎<嗙

萧炎利他蜓伞觊类，料敢，

萧炎焕缭胯掘砧庞玛味殒且

Epoch: 0, Batch: 0, Loss: 8.5821
萧炎屹帕。倒辰。列绌桥讪

萧炎榨羊腊奥栽钮魄狲霾层

萧炎涛婷嗔方煊铜龊诳荐的

萧炎朔户项副既祷滔髅角弥

萧炎装厉。镀茑箍阅滤锢匣

萧炎完顶塔涎位闲监爱残咧

萧炎老绷爆咫酒讹跟髪╰机

萧炎丘是盔的鸣夏坡牌驸盏

萧炎劾掷祜罗噎快纯傍袍渍

萧炎竟燎嫣哒祝礴秋视逸血

萧炎呆炙的在坎报攻坊姿臭

萧炎围们褛篓贬青曲拴始毁

Epoch: 0, Batch: 1, Loss: 7.7866
萧炎压岜耐爸柯芳栽。善ろ

萧炎这韵他优琐晾提.旬“

萧炎景派针雹赛遄瑰竖酸摩

萧炎毫苍牌才连紧颁颇狙胄

萧炎他氏眷清的憬拢塌,习

萧炎趸熏燕赶萱呢佼牌爸剑

萧炎铤孔的炽掺魁殴渚昨丽

萧炎咎过芬耄抿谓6我涟蛮

萧炎踮头踩擅崇置锊贝蒜兼

萧炎的其候托轱掖们馨也峥

萧炎飚黑从梏睬楼勋颓各介

萧炎娑风家的了堵方惩獗怂

Epoch: 0, Batch: 2, Loss: 7.7307
